In [1]:
!python utils.py

In [2]:
!python rdp_analysis.py

In [3]:
!pip install kymatio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 2.5 MB/s eta 0:00:00


In [4]:
!python MLModel.py

In [5]:
!python FLModel.py

In [6]:
from MLModel import *
from FLModel import *
from utils import *
from torchvision import datasets, transforms
import torch
import numpy as np
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")


In [7]:
scattering, K, (h, w) = get_scatter_transform()
scattering.to(device)

def get_scattered_feature(dataset):
    scatters = []
    targets = []

    loader = torch.utils.data.DataLoader(
        dataset, batch_size=256, shuffle=True, num_workers=1, pin_memory=True)


    for (data, target) in loader:
        data, target = data.to(device), target.to(device)
        if scattering is not None:
            data = scattering(data)
        scatters.append(data)
        targets.append(target)

    scatters = torch.cat(scatters, axis=0)
    targets = torch.cat(targets, axis=0)

    data = torch.utils.data.TensorDataset(scatters, targets)
    return data

def load_mnist(num_users):
    train = datasets.MNIST(root="~/data/", train=True, download=True, transform=transforms.ToTensor())
    test = datasets.MNIST(root="~/data/", train=False, download=True, transform=transforms.ToTensor())

    # get scattered features
    train = get_scattered_feature(train)
    test = get_scattered_feature(test)

    train_data = train[:][0].squeeze().cpu().float()
    train_label = train[:][1].cpu()

    test_data = test[:][0].squeeze().cpu().float()
    test_label = test[:][1].cpu()

    # split MNIST (training set) into non-iid data sets
    non_iid = []
    user_dict = mnist_noniid(train_label, num_users)
    for i in range(num_users):
        idx = user_dict[i]
        d = train_data[idx]
        targets = train_label[idx].float()
        non_iid.append((d, targets))
    non_iid.append((test_data.float(), test_label.float()))
    return non_iid


In [8]:
"""
1. load_data
2. generate clients (step 3)
3. generate aggregator
4. training
"""
client_num = 5
d = load_mnist(client_num)

torch.cuda.empty_cache()

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.52MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 159kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.31MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.70MB/s]


In [9]:
d[1][0][0].shape

torch.Size([81, 7, 7])

In [10]:
import warnings
warnings.filterwarnings("ignore")

lr = 0.075

fl_param = {
    'output_size': 10,
    'K': K,
    'h': h,
    'w': w,
    'client_num': client_num,
    'model': 'scatter',
    'data': d,
    'lr': lr,
    'E': 500,
    'C': 1,
    'eps': 4.0,
    'delta': 1e-5,
    'q': 0.01,
    'clip': 0.1,
    'tot_T': 10,
    'batch_size': 128,
    'device': device
}

fl_entity = FLServer(fl_param).to(device)

noise scale = 1.0771102905273438


In [11]:
import time
from sklearn.metrics import classification_report

# Initialize an empty list to store accuracy values across rounds
acc = []

# Record the start time of the training process
start_time = time.time()

# Loop over the total number of global iterations (communication rounds)
for t in range(fl_param['tot_T']):
    # Perform a global update and store the accuracy for the current round
    acc.append(fl_entity.global_update())

    # Evaluate the model on the test set to get predictions for classification report
    fl_entity.global_model.eval()
    all_preds = []
    all_true = []

    # Collect predictions and true labels
    for i in range(len(fl_entity.data)):
        t_pred_y = fl_entity.global_model(fl_entity.data[i])
        _, predicted = torch.max(t_pred_y, 1)
        all_preds.extend(predicted.cpu().numpy())  # Get the predicted labels
        all_true.extend(fl_entity.target[i].cpu().numpy())  # Get the true labels

    # Generate classification report
    report = classification_report(all_true, all_preds, digits=4)

    # Print the classification report along with the accuracy and time taken
    elapsed_time = time.time() - start_time
    print(f"Global Epoch {t + 1}/{fl_param['tot_T']}, Accuracy = {acc[-1]:.4f}, Time taken: {elapsed_time:.2f}s")
    print("Classification Report:")
    print(report)


Global Epoch 1/10, Accuracy = 0.8153, Time taken: 303.76s
Classification Report:
              precision    recall  f1-score   support

         0.0     0.9644    0.9673    0.9659       980
         1.0     0.9123    0.9894    0.9493      1135
         2.0     1.0000    0.0039    0.0077      1032
         3.0     0.4312    0.9921    0.6011      1010
         4.0     0.9322    0.9807    0.9558       982
         5.0     0.9754    0.6670    0.7923       892
         6.0     0.9886    0.9061    0.9455       958
         7.0     0.9521    0.9095    0.9303      1028
         8.0     0.7894    0.8234    0.8060       974
         9.0     0.9723    0.9049    0.9374      1009

    accuracy                         0.8153     10000
   macro avg     0.8918    0.8144    0.7891     10000
weighted avg     0.8909    0.8153    0.7877     10000

Global Epoch 2/10, Accuracy = 0.8790, Time taken: 600.94s
Classification Report:
              precision    recall  f1-score   support

         0.0     0.9405 